# * Re-Org Revenue 2025
(GX2, GX3, GX5, GX6)

In [7]:
import configparser
import datetime as dt
import pandas as pd
import numpy as np
import xlrd
import oracledb
import re

config = configparser.ConfigParser()
config.read('../../my_config.ini')
config.sections()

TDMDBPR_user = config['TDMDBPR']['username']
TDMDBPR_pwd = config['TDMDBPR']['password']
TDMDBPR_db = config['TDMDBPR']['db']
TDMDBPR_host = config['TDMDBPR']['host']
TDMDBPR_port = config['TDMDBPR']['port']

AKPIPRD_user = config['AKPIPRD']['username']
AKPIPRD_pwd = config['AKPIPRD']['password']
AKPIPRD_db = config['AKPIPRD']['db']
AKPIPRD_host = config['AKPIPRD']['host']
AKPIPRD_port = config['AKPIPRD']['port']

curr_dt = dt.datetime.now().date()
str_curr_dt = curr_dt.strftime('%Y%m%d')

In [8]:
''' Input parameter '''

op_dir = 'C:/Ruz/MyProject/Code/Jupyter/data/Adhoc/output'
op_file = f'revenue_monthly_{str_curr_dt}'

v_start_date = 20250101
v_end_date = 20251231
# v_curr_date = 20260430

print(f'\nParameter input...\n')
print(f'   -> op_dir: {op_dir}')
print(f'   -> op_file: {op_file}')
print(f'\n   -> v_start_date: {v_start_date}')
print(f'   -> v_end_date: {v_end_date}')
# print(f'   -> v_curr_date: {v_curr_date}')


Parameter input...

   -> op_dir: C:/Ruz/MyProject/Code/Jupyter/data/Adhoc/output
   -> op_file: revenue_monthly_20260430

   -> v_start_date: 20250101
   -> v_end_date: 20251231


## Import Transaction
-> AGG_PERF_NEWCO

In [9]:
''' Execute transaction '''


curr_datetime = dt.datetime.now().strftime('%Y-%m-%d, %H:%M:%S')
print(f'\nData as of {curr_datetime}')


# Connect : TDMDBPR
src_dsn = f'{TDMDBPR_user}/{TDMDBPR_pwd}@{TDMDBPR_host}:{TDMDBPR_port}/{TDMDBPR_db}'
src_conn = oracledb.connect(src_dsn)
src_cur = src_conn.cursor()


query = (f"""
	WITH W_PARAM AS
	(
		SELECT {v_start_date} AS V_INI_DAY_START
			, {v_end_date} AS V_INI_DAY_END
		FROM DUAL
	) -->> W_PARAM
	-----------------------------------------------------------------------------------------------------------------------


	, W_ORG AS 
	( 
		SELECT DISTINCT ZONE_TYPE
			, ORGID_G, TDS_SGMD
			, ORGID_H, HOP_HINT
		FROM CDSAPPO.DIM_MOOC_AREA
		WHERE TEAM_CODE <> 'ไม่ระบุ' AND REMARK <> 'Dummy'
		AND ORGID_G IN ('GX2' ,'GX3' ,'GX5', 'GX6') --Re-Org
	) -->> W_ORG
	-----------------------------------------------------------------------------------------------------------------------


	, W_ACTUAL_ORIGINAL_2025 AS 
	(
		SELECT TM_KEY_MTH, TM_KEY_DAY, PRODUCT_GRP, METRIC_CD, METRIC_NAME, COMP_CD, AREA_TYPE, AREA_CD, AREA_NAME
			, ACTUAL_SNAP, TARGET_SNAP, ACH_SNAP, PPN_TM
		FROM GEOSPCAPPO.AGG_PERF_NEWCO
		WHERE METRIC_CD IN (
			'B2R010100' --Postpaid Revenue B2C
			, 'DB2R010100' --Postpaid Revenue B2C : DTAC
			, 'TB2R010100' --Postpaid Revenue B2C : TMH
			, 'B1R000100' --Prepaid Revenue
			, 'DB1R000100' --Prepaid Revenue : DTAC
			, 'TB1R000100' --Prepaid Revenue : TMH
			, 'TB3R000100' --TOL Revenue
			, 'TB4R000100' --TVS Revenue
			)
		AND AREA_TYPE = 'G'
		AND AREA_CD NOT IN ('GX2' ,'GX3' ,'GX5', 'GX6') --Re-Org
		AND TM_KEY_DAY BETWEEN (SELECT V_INI_DAY_START FROM W_PARAM) AND (SELECT V_INI_DAY_END FROM W_PARAM)
	) -->> W_ACTUAL_ORIGINAL_2025
	-----------------------------------------------------------------------------------------------------------------------


	, W_ACTUAL_TEMP AS 
	(
		-->> Actual 2025
		SELECT TM_KEY_MTH, TM_KEY_DAY, PRODUCT_GRP, METRIC_CD, METRIC_NAME, COMP_CD, AREA_TYPE, AREA_CD, AREA_NAME
			, ACTUAL_SNAP, TARGET_SNAP
			, ACTUAL_SNAP / TARGET_SNAP * 100 AS ACH
			, SYSDATE AS LOAD_DATE
		FROM (
			SELECT A.TM_KEY_MTH, A.TM_KEY_DAY, A.PRODUCT_GRP, A.METRIC_CD, A.METRIC_NAME, A.COMP_CD
				, 'G' AS AREA_TYPE, O.ORGID_G AS AREA_CD, O.TDS_SGMD AS AREA_NAME
				, SUM(A.ACTUAL_SNAP) AS ACTUAL_SNAP, SUM(A.TARGET_SNAP) AS TARGET_SNAP
			FROM (
				SELECT TM_KEY_MTH, TM_KEY_DAY, PRODUCT_GRP, METRIC_CD, METRIC_NAME, COMP_CD, AREA_TYPE, AREA_CD, AREA_NAME, ACTUAL_SNAP, TARGET_SNAP, PPN_TM
				FROM GEOSPCAPPO.AGG_PERF_NEWCO
				WHERE METRIC_CD IN (
					'DB2R010100' --Postpaid Revenue B2C : DTAC
					, 'TB2R010100' --Postpaid Revenue B2C : TMH
					, 'DB1R000100' --Prepaid Revenue : DTAC
					, 'TB1R000100' --Prepaid Revenue : TMH
					, 'TB3R000100' --TOL Revenue
					, 'TB4R000100' --TVS Revenue
					)
				AND AREA_TYPE = 'H'
				AND TM_KEY_DAY BETWEEN (SELECT V_INI_DAY_START FROM W_PARAM) AND (SELECT V_INI_DAY_END FROM W_PARAM)
			) A
			INNER JOIN W_ORG O
				ON O.ORGID_H = A.AREA_CD
			GROUP BY A.TM_KEY_MTH, A.TM_KEY_DAY, A.PRODUCT_GRP, A.METRIC_CD, A.METRIC_NAME, A.COMP_CD, O.ORGID_G, O.TDS_SGMD
		) TMP
		
		UNION ALL 
		
		-->> Actual 2026
		SELECT TM_KEY_MTH, TM_KEY_DAY, PRODUCT_GRP, METRIC_CD, METRIC_NAME, COMP_CD, AREA_TYPE, AREA_CD, AREA_NAME
			, CASE 	WHEN METRIC_CD = 'DB1R000100' AND TM_KEY_DAY = 20260327
					THEN SUM(ACTUAL_SNAP) OVER(PARTITION BY METRIC_CD, AREA_CD, TM_KEY_MTH) / 30
					ELSE ACTUAL_SNAP 
					END ACTUAL_SNAP
			, CASE 	WHEN METRIC_CD = 'DB1R000100' AND TM_KEY_DAY = 20260327
					THEN SUM(TARGET_SNAP) OVER(PARTITION BY METRIC_CD, AREA_CD, TM_KEY_MTH) / 30
					ELSE TARGET_SNAP 
					END TARGET_SNAP
			, ACH_SNAP, PPN_TM
		FROM GEOSPCAPPO.AGG_PERF_NEWCO
		WHERE METRIC_CD IN (
			'DB2R010100' --Postpaid Revenue B2C : DTAC
			, 'TB2R010100' --Postpaid Revenue B2C : TMH
			, 'DB1R000100' --Prepaid Revenue : DTAC
			, 'TB1R000100' --Prepaid Revenue : TMH
			, 'TB3R000100' --TOL Revenue
			, 'TB4R000100' --TVS Revenue
			)
		AND AREA_TYPE = 'G'
		AND TM_KEY_DAY BETWEEN 20260101 AND 20260331
	) -->> W_ACTUAL_TEMP
	-----------------------------------------------------------------------------------------------------------------------


	, W_ACTUAL_NEW AS 
	(
		-->> 'B2R010100' --Postpaid Revenue B2C
		SELECT TM_KEY_MTH, TM_KEY_DAY, PRODUCT_GRP
			, 'B2R010100' AS METRIC_CD, 'Postpaid Revenue B2C' AS METRIC_NAME, 'ALL' AS COMP_CD
			, AREA_TYPE, AREA_CD, AREA_NAME
			, SUM(ACTUAL_SNAP) AS ACTUAL_SNAP, SUM(TARGET_SNAP) AS TARGET_SNAP
			, SUM(ACTUAL_SNAP) / SUM(TARGET_SNAP) * 100 AS ACH
			, SYSDATE AS LOAD_DATE
		FROM W_ACTUAL_TEMP 
		WHERE METRIC_CD IN (	
			'DB2R010100' --Postpaid Revenue B2C : DTAC
			, 'TB2R010100' --Postpaid Revenue B2C : TMH
			)
		GROUP BY TM_KEY_MTH, TM_KEY_DAY, PRODUCT_GRP, AREA_TYPE, AREA_CD, AREA_NAME
		
		UNION ALL 
		
		-->> 'B1R000100' --Prepaid Revenue
		SELECT TM_KEY_MTH, TM_KEY_DAY, PRODUCT_GRP
			, 'B1R000100' AS METRIC_CD, 'Prepaid Revenue' AS METRIC_NAME, 'ALL' AS COMP_CD
			, AREA_TYPE, AREA_CD, AREA_NAME
			, SUM(ACTUAL_SNAP) AS ACTUAL_SNAP, SUM(TARGET_SNAP) AS TARGET_SNAP
			, SUM(ACTUAL_SNAP) / SUM(TARGET_SNAP) * 100 AS ACH
			, SYSDATE AS LOAD_DATE
		FROM W_ACTUAL_TEMP 
		WHERE METRIC_CD IN (	
			'DB1R000100' --Prepaid Revenue : DTAC
			, 'TB1R000100' --Prepaid Revenue : TMH
			)
		GROUP BY TM_KEY_MTH, TM_KEY_DAY, PRODUCT_GRP, AREA_TYPE, AREA_CD, AREA_NAME
		
		UNION ALL 
		
		SELECT * FROM W_ACTUAL_TEMP
	) -->> W_ACTUAL_NEW
	-----------------------------------------------------------------------------------------------------------------------


	, W_DAILY_REV AS 
	(
		SELECT * FROM W_ACTUAL_NEW
		
		UNION ALL 
		
		SELECT * FROM W_ACTUAL_ORIGINAL_2025
	) -->> W_DAILY_REV
	-----------------------------------------------------------------------------------------------------------------------
	-----------------------------------------------------------------------------------------------------------------------


	-->> Monthly Revenue

	SELECT /*+PARALLEL(8)*/ 
		TM_KEY_MTH, PRODUCT_GRP, METRIC_CD, METRIC_NAME, COMP_CD, AREA_TYPE, AREA_CD, AREA_NAME
		, SUM(ACTUAL_SNAP) AS MTD_ACTUAL, SUM(TARGET_SNAP) AS MTD_TARGET
		, SUM(ACTUAL_SNAP) / SUM(TARGET_SNAP) * 100 AS MTD_ACH
		, MAX(LOAD_DATE) AS LOAD_DATE
	FROM W_DAILY_REV
	GROUP BY TM_KEY_MTH, PRODUCT_GRP, METRIC_CD, METRIC_NAME, COMP_CD, AREA_TYPE, AREA_CD, AREA_NAME
	--ORDER BY TM_KEY_MTH, PRODUCT_GRP, METRIC_CD, AREA_CD
""")


try:
    # Create Dataframe
    src_cur.execute(query)
    rows = src_cur.fetchall()
    src_df = pd.DataFrame.from_records(rows, columns=[x[0] for x in src_cur.description])
    print(f'\nDataFrame: {src_df.shape[0]} rows, {src_df.shape[1]} columns')
    src_cur.close()


except oracledb.DatabaseError as e:
    print(f'\nError with Oracle : {e}')


finally:
    src_conn.close()


Data as of 2026-04-30, 21:54:11

DataFrame: 960 rows, 12 columns


## Rawdata

In [10]:
src_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 960 entries, 0 to 959
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   TM_KEY_MTH   960 non-null    int64         
 1   PRODUCT_GRP  960 non-null    object        
 2   METRIC_CD    960 non-null    object        
 3   METRIC_NAME  960 non-null    object        
 4   COMP_CD      960 non-null    object        
 5   AREA_TYPE    960 non-null    object        
 6   AREA_CD      960 non-null    object        
 7   AREA_NAME    960 non-null    object        
 8   MTD_ACTUAL   960 non-null    float64       
 9   MTD_TARGET   960 non-null    float64       
 10  MTD_ACH      960 non-null    float64       
 11  LOAD_DATE    960 non-null    datetime64[ns]
dtypes: datetime64[ns](1), float64(3), int64(1), object(7)
memory usage: 90.1+ KB


In [11]:
rawdata_df = src_df.sort_values(by=['TM_KEY_MTH', 'PRODUCT_GRP', 'METRIC_CD', 'AREA_CD']).reset_index(drop=True)
rawdata_df

,TM_KEY_MTH,PRODUCT_GRP,METRIC_CD,METRIC_NAME,COMP_CD,AREA_TYPE,AREA_CD,AREA_NAME,MTD_ACTUAL,MTD_TARGET,MTD_ACH,LOAD_DATE
0,202501,Postpaid,B2R010100,Postpaid Revenue B2C,ALL,G,GX1,BMA : West,9.394973e+08,8.997438e+08,104.418310,2026-04-30 02:45:31
1,202501,Postpaid,B2R010100,Postpaid Revenue B2C,ALL,G,GX2,BMA-East,1.401202e+09,1.343033e+09,104.331156,2026-04-30 21:54:15
2,202501,Postpaid,B2R010100,Postpaid Revenue B2C,ALL,G,GX3,East,7.354405e+08,6.776122e+08,108.534127,2026-04-30 21:54:15
3,202501,Postpaid,B2R010100,Postpaid Revenue B2C,ALL,G,GX4,North,5.966374e+08,6.457404e+08,92.395861,2026-04-30 02:45:31
4,202501,Postpaid,B2R010100,Postpaid Revenue B2C,ALL,G,GX5,Northeast1,3.923591e+08,4.547789e+08,86.274692,2026-04-30 21:54:15
...,...,...,...,...,...,...,...,...,...,...,...,...
955,202603,TVS,TB4R000100,TVS Revenue,TRUE,G,GX4,North,1.340780e+07,1.858484e+07,72.143779,2026-04-30 02:45:31
956,202603,TVS,TB4R000100,TVS Revenue,TRUE,G,GX5,Northeast 1,6.737051e+06,9.475724e+06,71.098002,2026-04-30 02:45:31
957,202603,TVS,TB4R000100,TVS Revenue,TRUE,G,GX6,Northeast 2,1.050613e+07,1.456367e+07,72.139323,2026-04-30 02:45:31
958,202603,TVS,TB4R000100,TVS Revenue,TRUE,G,GX7,"Central, West",1.647220e+07,2.267553e+07,72.643068,2026-04-30 02:45:31


## Output file

In [12]:
''' Generate CSV file '''

rawdata_df.to_csv(f'{op_dir}/{op_file}.csv', index=False, encoding='utf-8')
print(f'\n   -> Generate "{op_file}.csv" successfully')


   -> Generate "revenue_monthly_20260430.csv" successfully
